[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_2_opencv_foundations/04_intro_to_opencv.ipynb)

# Notebook 04 — Introduction to OpenCV
### 6D Pose Vision Workshop · Part 2: OpenCV Foundations

**Estimated time:** 40 minutes  
**Dependencies:** `opencv-python`, `opencv-contrib-python`, `numpy`, `matplotlib`

---

## Recommended Watch

> Watch these **before** opening this notebook. VN27 gives you the conceptual map (what is CV vs image processing?), VN25 shows you everything OpenCV can do in 5 minutes, and VN26 explains the `import cv2` naming quirk that trips up every beginner.

| Title | Link | Duration |
|---|---|---|
| **Image Processing VS Computer Vision: What's The Difference?** | [▶ Watch](https://www.youtube.com/watch?v=pcxhj5KFI6M) | ~8 min |
| **OpenCV Tutorial in 5 Minutes — All Modules Overview** | [▶ Watch](https://www.youtube.com/watch?v=PeMM80WimN4) | ~5 min |
| **How to Install OpenCV for Python** | [▶ Watch](https://youtu.be/M6jukmppMqU) | ~10 min |

> **Suggested order:** VN27 → VN25 → VN26. Start with the concept (what IS computer vision?), then get the landscape (what can OpenCV do?), then the setup gotcha (`pip install opencv-python` but `import cv2`).

---

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install opencv-python opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab — packages installed")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version:  {np.__version__}")

# Build info: check for CUDA support
build_info = cv2.getBuildInformation()
cuda_line = [l for l in build_info.split('\n') if 'CUDA' in l and 'Support' in l]
if cuda_line:
    print(f"CUDA support:   {cuda_line[0].strip()}")
else:
    print("CUDA support:   Not detected in this OpenCV build (CPU-only build is fine for Parts 1–6)")

## Learning Objectives

By the end of this notebook you will:

- Know what OpenCV is and why it's the standard library for computer vision
- Understand the **BGR vs RGB** difference and how to handle it
- Read, write, and display images with `cv2.imread` / `cv2.imwrite`
- Capture video frames from a file or webcam with `cv2.VideoCapture`
- Draw shapes and text on images with OpenCV's drawing functions
- Understand `cv2.waitKey` and why it's needed for interactive windows

---

## 1. What Is OpenCV?

**OpenCV** (Open Source Computer Vision Library) is the most widely used computer vision library in the world. Originally written in C++ by Intel in 1999, it has Python bindings that let us use its high-performance algorithms from Python.

### What it does

| Category | What OpenCV provides |
|---|---|
| Image I/O | Read/write images (JPEG, PNG, TIFF, BMP, ...) and video |
| Image processing | Resize, filter, transform, threshold, morph |
| Feature detection | SIFT, ORB, FAST, Harris corners, ArUco markers |
| Camera calibration | Find intrinsics K, distortion coefficients |
| Pose estimation | solvePnP, ArUco pose, essential/fundamental matrices |
| 3D vision | Stereo disparity, depth, point clouds |
| Video analysis | Optical flow, background subtraction, tracking |
| Deep learning | `cv2.dnn` module — load ONNX, TensorFlow, Caffe models |

### Why OpenCV and not PyTorch/scikit-image?

- **Speed**: Core operations are implemented in C++ and heavily optimized
- **Camera math is built-in**: `calibrateCamera`, `solvePnP`, `estimatePoseSingleMarkers` are all ready to use
- **Industry standard**: robotics teams, industrial vision systems, research labs all use it
- **Real-time capable**: processes 1080p video at 60+ FPS for many operations

### Two packages

```bash
pip install opencv-python          # core OpenCV
pip install opencv-contrib-python  # core + extras (ArUco, SIFT, SURF, ...)
```

We need `opencv-contrib-python` for ArUco marker detection in Part 5. The `contrib` package **replaces** `opencv-python` — don't install both.

### GPU-accelerated OpenCV (for Part 7+)

The pip wheels are **CPU-only**. For GPU acceleration, you build OpenCV from source with CUDA enabled — see **[VN8](https://youtu.be/HsuKxjQhFU0)** or **[VN32](https://youtu.be/d8Jx6zO1yw0)** for the full process. For Parts 1–6, the CPU build is fully sufficient.

---

## 2. BGR vs RGB — The Most Common Beginner Mistake

### Why OpenCV uses BGR

When OpenCV was created, many hardware cameras output **BGR** byte order. OpenCV adopted this convention and has kept it for backwards compatibility ever since.

**Every time you call `cv2.imread()` or `cv2.VideoCapture.read()`, you get BGR.**

Most other libraries (matplotlib, PIL/Pillow, PyTorch's torchvision, scikit-image) use **RGB**.

### What goes wrong if you don't convert

```
OpenCV loads image as:  [B=0, G=0, R=255] ← a pure red pixel
matplotlib sees it as:  [R=0, G=0, B=255] ← interprets as pure blue!
```

### The three ways to convert

In [ ]:
# Three ways to convert BGR ↔ RGB

import cv2
import numpy as np

# Simulate a BGR image with a red pixel
# In BGR: red is channel index 2
bgr = np.array([[[0, 0, 255]]], dtype=np.uint8)  # shape (1, 1, 3): one red pixel
print(f"Original BGR pixel:  {bgr[0,0]}  (Blue=0, Green=0, Red=255)")

# Method 1: NumPy slice reversal — fastest, works anywhere
rgb_1 = bgr[:, :, ::-1]
print(f"Method 1 (slice):    {rgb_1[0,0]}  (Red=255, Green=0, Blue=0)")

# Method 2: cv2.cvtColor — explicit, readable, same result
rgb_2 = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
print(f"Method 2 (cvtColor): {rgb_2[0,0]}")

# Method 3: numpy indexing
rgb_3 = bgr[:, :, [2, 1, 0]]  # reorder channels: take R(2), G(1), B(0)
print(f"Method 3 (indexing): {rgb_3[0,0]}")

print("\nAll three methods produce identical results:", 
      np.all(rgb_1 == rgb_2) and np.all(rgb_2 == rgb_3))

## 3. Reading and Writing Images

In [ ]:
# First, create a sample image to work with (since we may not have one on disk)
import os

# Create assets/images directory if it doesn't exist
os.makedirs('../assets/images', exist_ok=True)

# Generate a synthetic 'robot scene' image
def create_sample_image(height=480, width=640):
    """Create a synthetic scene resembling an industrial environment."""
    img = np.full((height, width, 3), 180, dtype=np.uint8)  # gray background
    
    # Floor (darker gray)
    img[height//2:, :] = 100
    
    # Wall boundary line
    cv2.line(img, (0, height//2), (width, height//2), (50, 50, 50), 2)
    
    # "Pallet" on floor (brown rectangle)
    cv2.rectangle(img, (150, height//2 + 50), (400, height//2 + 150), (20, 80, 120), -1)
    cv2.rectangle(img, (150, height//2 + 50), (400, height//2 + 150), (0, 0, 0), 2)
    
    # Pallet openings
    cv2.rectangle(img, (190, height//2 + 80), (270, height//2 + 120), (60, 40, 20), -1)
    cv2.rectangle(img, (290, height//2 + 80), (370, height//2 + 120), (60, 40, 20), -1)
    
    # "ArUco marker" placeholder on wall (white square with black border)
    cv2.rectangle(img, (460, 80), (580, 200), (255, 255, 255), -1)
    cv2.rectangle(img, (460, 80), (580, 200), (0, 0, 0), 4)
    # Inner pattern
    cv2.rectangle(img, (480, 100), (510, 130), (0, 0, 0), -1)
    cv2.rectangle(img, (530, 150), (560, 180), (0, 0, 0), -1)
    
    # Label
    cv2.putText(img, 'Sample Robot Scene', (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    
    return img

sample_img = create_sample_image()
sample_path = '../assets/images/sample_scene.jpg'
cv2.imwrite(sample_path, sample_img)
print(f"Sample image created: {sample_path}")
print(f"Shape: {sample_img.shape}, dtype: {sample_img.dtype}")

In [ ]:
# cv2.imread — reading images

# Load in color (default) — returns BGR uint8 array
img_color = cv2.imread(sample_path)  # or cv2.IMREAD_COLOR

# Load as grayscale
img_gray = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)

# Load with alpha channel (if PNG with transparency)
# img_alpha = cv2.imread('image.png', cv2.IMREAD_UNCHANGED)  # shape: (H, W, 4)

print(f"Color image:  shape={img_color.shape}, dtype={img_color.dtype}")
print(f"Gray image:   shape={img_gray.shape},  dtype={img_gray.dtype}")

# IMPORTANT: imread returns None if file not found (not an exception!)
bad_img = cv2.imread('nonexistent.jpg')
print(f"Missing file: {bad_img}  ← None, no exception raised")

# Always check for None:
def safe_imread(path, flags=cv2.IMREAD_COLOR):
    """Load image with error checking."""
    img = cv2.imread(path, flags)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")
    return img

# Display the loaded images
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_color[:, :, ::-1])  # BGR → RGB
axes[0].set_title('Color (BGR→RGB)')
axes[0].axis('off')
axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title('Grayscale')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# cv2.imwrite — saving images

# Save as JPEG (quality 0-100, default ~95)
cv2.imwrite('../assets/images/scene_quality_80.jpg', img_color, [cv2.IMWRITE_JPEG_QUALITY, 80])

# Save as PNG (lossless, compression 0-9, default 3)
cv2.imwrite('../assets/images/scene_lossless.png', img_color, [cv2.IMWRITE_PNG_COMPRESSION, 3])

# Save grayscale
cv2.imwrite('../assets/images/scene_gray.png', img_gray)

print("Files saved:")
for fname in ['scene_quality_80.jpg', 'scene_lossless.png', 'scene_gray.png']:
    path = f'../assets/images/{fname}'
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname}: {size_kb:.1f} KB")

print("\nNote: PNG is larger (lossless), JPEG is smaller (lossy compression)")

## 4. Displaying Images

### Two approaches

| Method | When to use | Notes |
|---|---|---|
| `cv2.imshow()` | Local scripts, GUI windows | Opens a separate window; blocked in Colab |
| `plt.imshow()` | Notebooks (Jupyter, Colab) | Inline; requires BGR→RGB conversion |

### cv2.imshow (for local interactive scripts)

```python
cv2.imshow('Window Title', img)   # display BGR image (no conversion needed!)
cv2.waitKey(0)                    # wait indefinitely for a key press
# cv2.waitKey(30)                 # wait 30ms (for video loops ~33 FPS)
cv2.destroyAllWindows()           # close all OpenCV windows
```

> `cv2.imshow` accepts **BGR** directly. No conversion needed for display.

We define a unified helper below that works in both environments:

In [ ]:
# Unified show_image function — use this throughout the course

def show_image(img, title='', figsize=(10, 7), cmap=None):
    """
    Display a BGR (OpenCV) or grayscale image inline in notebooks.
    Works identically in VSCode and Google Colab.
    """
    plt.figure(figsize=figsize)
    
    if img.ndim == 3:
        plt.imshow(img[:, :, ::-1])  # BGR → RGB
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    
    plt.title(title, pad=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_images(pairs, figsize_per=(6, 5)):
    """
    Display multiple (image, title) pairs side by side.
    Each image can be BGR color or grayscale.
    """
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0] * n, figsize_per[1]))
    if n == 1:
        axes = [axes]
    
    for ax, (img, title) in zip(axes, pairs):
        if img.ndim == 3:
            ax.imshow(img[:, :, ::-1])
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()


# Test the helpers
show_images([
    (img_color, 'Color (BGR auto-converted)'),
    (img_gray,  'Grayscale')
])

## 5. Video Capture

### VideoCapture — reading from files or cameras

```python
# From a file:
cap = cv2.VideoCapture('video.mp4')

# From a webcam (index 0 = first camera):
cap = cv2.VideoCapture(0)

# From an IP camera (RTSP stream):
cap = cv2.VideoCapture('rtsp://192.168.1.10/stream')
```

### The video capture loop pattern

In [ ]:
# Standard video capture loop — we simulate it here with a synthetic video

def simulate_video_loop(num_frames=5):
    """
    Demonstrates the VideoCapture loop pattern by simulating frames.
    In a real script, replace with: cap = cv2.VideoCapture(0)
    """
    # The real loop would be:
    # cap = cv2.VideoCapture(0)  # or VideoCapture('video.mp4')
    # if not cap.isOpened():
    #     raise IOError("Cannot open camera or video file")

    frames_collected = []

    for frame_idx in range(num_frames):
        # Real version: ret, frame = cap.read()
        # Simulated: generate a frame with frame number
        frame = np.full((240, 320, 3), 180, dtype=np.uint8)
        cv2.putText(frame, f'Frame {frame_idx + 1}/{num_frames}',
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 50), 2)
        cv2.circle(frame, (160 + frame_idx * 15, 120), 20, (0, 0, 255), -1)
        
        ret = True  # in real code: ret is False when video ends

        if not ret:
            print("Video ended")
            break

        # === YOUR PROCESSING GOES HERE ===
        # e.g., detect ArUco markers, run pose estimation, etc.
        # =================================

        frames_collected.append(frame.copy())

        # In a real window-based loop:
        # cv2.imshow('Live', frame)
        # key = cv2.waitKey(30)  # 30ms = ~33 FPS
        # if key == ord('q'):    # press Q to quit
        #     break

    # Real version: cap.release(); cv2.destroyAllWindows()
    return frames_collected

frames = simulate_video_loop(5)
print(f"Captured {len(frames)} frames, each shape: {frames[0].shape}")

# Display all frames
fig, axes = plt.subplots(1, len(frames), figsize=(16, 3))
for ax, frame in zip(axes, frames):
    ax.imshow(frame[:, :, ::-1])
    ax.axis('off')
plt.suptitle('Simulated video frames (object moving right)')
plt.tight_layout()
plt.show()

In [ ]:
# Reading video properties — useful for processing existing video files

# Simulate what you'd do with a real video file
print("VideoCapture properties you'll commonly query:")
print()
props = {
    'cv2.CAP_PROP_FRAME_WIDTH':  (cv2.CAP_PROP_FRAME_WIDTH,  'Frame width (pixels)'),
    'cv2.CAP_PROP_FRAME_HEIGHT': (cv2.CAP_PROP_FRAME_HEIGHT, 'Frame height (pixels)'),
    'cv2.CAP_PROP_FPS':          (cv2.CAP_PROP_FPS,          'Frames per second'),
    'cv2.CAP_PROP_FRAME_COUNT':  (cv2.CAP_PROP_FRAME_COUNT,  'Total frame count'),
    'cv2.CAP_PROP_POS_FRAMES':   (cv2.CAP_PROP_POS_FRAMES,   'Current frame position'),
}

for name, (code, description) in props.items():
    print(f"  cap.get({name})")
    print(f"    → {description}")
    print()

print("Example usage:")
print("  cap = cv2.VideoCapture('video.mp4')")
print("  w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))")
print("  h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))")
print("  fps = cap.get(cv2.CAP_PROP_FPS)")
print("  print(f'Video: {w}x{h} @ {fps} FPS')")

## 6. Drawing Functions

OpenCV's drawing functions modify the image **in-place**. Always work on a **copy** if you want to preserve the original:

```python
canvas = img.copy()   # modify canvas, original untouched
```

### Common drawing functions

```python
# All colors are BGR tuples (not RGB!)

cv2.rectangle(img, pt1, pt2, color, thickness)   # thickness=-1 → filled
cv2.circle(img, center, radius, color, thickness)
cv2.line(img, pt1, pt2, color, thickness)
cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.polylines(img, pts, isClosed, color, thickness)
cv2.putText(img, text, org, fontFace, fontScale, color, thickness)
```

In [ ]:
# Drawing demo — visualizations we'll use throughout the course

canvas = np.full((400, 600, 3), 240, dtype=np.uint8)  # light gray background

# 1. Rectangle — outline (bounding box style)
cv2.rectangle(canvas, (20, 20), (180, 120), (0, 0, 200), 2)  # red outline
cv2.putText(canvas, 'rectangle (outline)', (20, 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 1)

# 2. Rectangle — filled
cv2.rectangle(canvas, (200, 20), (360, 120), (200, 100, 0), -1)  # filled blue
cv2.putText(canvas, 'rectangle (filled)', (200, 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

# 3. Circle
cv2.circle(canvas, (480, 70), 45, (0, 150, 0), 3)  # green circle
cv2.putText(canvas, 'circle', (450, 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 150, 0), 1)

# 4. Line
cv2.line(canvas, (20, 160), (580, 160), (100, 100, 100), 1)  # horizontal divider

# 5. Text (various sizes and colors)
fonts = [
    (cv2.FONT_HERSHEY_SIMPLEX,  'SIMPLEX'),
    (cv2.FONT_HERSHEY_DUPLEX,   'DUPLEX'),
    (cv2.FONT_HERSHEY_COMPLEX,  'COMPLEX'),
]
for i, (font, name) in enumerate(fonts):
    cv2.putText(canvas, f'{name}: Pose=1.5m 12deg', 
                (20, 200 + i * 50), font, 0.65, (50, 50, 50), 1)

# 6. Polylines — draw an axis frame (like we'll see in pose estimation!)
origin = (450, 300)
pts_x = np.array([origin, (530, 270)], dtype=np.int32)
pts_y = np.array([origin, (420, 250)], dtype=np.int32)
pts_z = np.array([origin, (450, 200)], dtype=np.int32)
cv2.arrowedLine(canvas, origin, (530, 270), (0, 0, 255), 2, tipLength=0.3)  # X = red
cv2.arrowedLine(canvas, origin, (380, 280), (0, 255, 0), 2, tipLength=0.3)  # Y = green
cv2.arrowedLine(canvas, origin, (450, 200), (255, 0, 0), 2, tipLength=0.3)  # Z = blue
cv2.putText(canvas, 'X', (535, 275), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
cv2.putText(canvas, 'Y', (365, 285), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
cv2.putText(canvas, 'Z', (455, 195), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
cv2.putText(canvas, 'Coordinate axes', (370, 385),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

show_image(canvas, 'OpenCV Drawing Functions')

## 7. cv2.waitKey — Why It Matters

`cv2.waitKey(ms)` does two things:
1. Waits for `ms` milliseconds before returning
2. Returns the key code of any key pressed during that time (-1 if none)

It's required because `cv2.imshow` only **schedules** the window to be drawn — the actual drawing happens during the `waitKey` call.

```python
# Common patterns:
cv2.waitKey(0)          # wait indefinitely — for single image display
cv2.waitKey(1)          # wait 1ms — use in tight loops (max FPS)
cv2.waitKey(30)         # wait 30ms — ~33 FPS cap
cv2.waitKey(1000)       # wait 1 second

# Key detection:
key = cv2.waitKey(30) & 0xFF   # & 0xFF normalizes across platforms
if key == ord('q'):   # Q to quit
    break
if key == ord('s'):   # S to save
    cv2.imwrite('frame.jpg', frame)
if key == 27:         # Escape key (ASCII 27)
    break
```

**Note:** `& 0xFF` masks the key code to 8 bits. On some Linux systems without it, key codes include extra bits that make `key == ord('q')` always False.

In [ ]:
# Demonstrate key code values (without actually opening a window)

common_keys = {
    'q': ord('q'),
    'Q': ord('Q'),
    's': ord('s'),
    'r': ord('r'),
    'Escape': 27,
    'Space': 32,
    'Enter': 13,
}

print("Common key codes for cv2.waitKey():")
for key_name, code in common_keys.items():
    print(f"  '{key_name}': {code}")

print("\nFull interactive loop template:")
template = '''
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Cannot open camera")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # === Process frame here ===
        # e.g., detect markers, run pose estimation
        # ==========================

        cv2.imshow('Camera', frame)
        key = cv2.waitKey(30) & 0xFF

        if key == ord('q') or key == 27:   # Q or Escape to quit
            break
        elif key == ord('s'):               # S to save current frame
            cv2.imwrite('saved_frame.jpg', frame)
            print("Frame saved!")
finally:
    cap.release()              # ALWAYS release, even if error occurred
    cv2.destroyAllWindows()
'''
print(template)

## Recap

| Concept | Key takeaway |
|---|---|
| OpenCV | The standard CV library — camera math, filters, feature detection, pose estimation |
| Two packages | `opencv-python` (core) vs `opencv-contrib-python` (core + ArUco, SIFT, etc.) |
| BGR | OpenCV stores color as Blue-Green-Red (not RGB) — flip with `[:,:,::-1]` or `cvtColor` |
| `cv2.imread` | Returns BGR uint8 array, or **None** if file not found (not an exception!) |
| `cv2.imwrite` | Saves to disk; file format inferred from extension |
| Display | `cv2.imshow` for local; `plt.imshow(img[:,:,::-1])` for notebooks/Colab |
| VideoCapture | `cap = VideoCapture(0)` → `ret, frame = cap.read()` → `cap.release()` |
| Drawing | All functions modify in-place; use `.copy()` to preserve original |
| `waitKey` | Required for `imshow` to actually render; also detects key presses |

---

**Next:** `05_image_operations.ipynb` — resize, color spaces, filters, edge detection, and building a preprocessing pipeline.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Load, convert, and annotate
# ============================================================
# 1. Load the sample_scene.jpg image
# 2. Convert it to RGB (for matplotlib display)
# 3. Draw a bright green rectangle around the 'pallet' region
#    (rough location: rows 290-390, cols 150-400)
# 4. Add the text "PALLET DETECTED" above the rectangle
# 5. Display the annotated image
#
# Note: rectangle and text coords are in BGR image (col, row) = (x, y)

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img = cv2.imread('../assets/images/sample_scene.jpg')
# annotated = img.copy()
#
# # Draw bounding box (green, thickness 3)
# cv2.rectangle(annotated, (150, 290), (400, 390), (0, 255, 0), 3)
#
# # Add label
# cv2.putText(annotated, 'PALLET DETECTED',
#             (150, 285), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
#
# show_image(annotated, 'Annotated Scene')

In [ ]:
# ============================================================
# EXERCISE 2: Video frame analysis function
# ============================================================
# Write a function process_frame(frame) that:
#   - Takes a BGR frame as input
#   - Draws a crosshair at the center of the frame
#   - Displays the frame number and pixel value at center in the top-left
#   - Returns the annotated frame
#
# Test it by applying to 3 synthetic frames and displaying results.

# YOUR CODE HERE
def process_frame(frame, frame_num=0):
    pass  # replace with your implementation


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def process_frame(frame, frame_num=0):
#     out = frame.copy()
#     h, w = out.shape[:2]
#     cx, cy = w // 2, h // 2
#
#     # Crosshair
#     cv2.line(out, (cx - 20, cy), (cx + 20, cy), (0, 255, 0), 1)
#     cv2.line(out, (cx, cy - 20), (cx, cy + 20), (0, 255, 0), 1)
#     cv2.circle(out, (cx, cy), 5, (0, 255, 0), 1)
#
#     # Info text
#     center_val = frame[cy, cx].tolist()  # BGR values at center
#     info = f'Frame:{frame_num} | Center BGR:{center_val}'
#     cv2.putText(out, info, (10, 25),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)
#
#     return out
#
# # Test with 3 synthetic frames
# test_frames = []
# for i in range(3):
#     f = np.full((240, 320, 3), (i * 60, 100, 200 - i * 50), dtype=np.uint8)
#     test_frames.append(process_frame(f, frame_num=i))
#
# show_images([(f, f'Frame {i}') for i, f in enumerate(test_frames)])

In [ ]:
# ============================================================
# EXERCISE 3: Safe image loader with metadata
# ============================================================
# Write a function load_image_info(path) that:
#   - Safely loads an image (handles missing file gracefully)
#   - Returns a dict with keys: 'image', 'path', 'shape', 'dtype',
#     'size_kb', 'channels', 'width', 'height'
#   - Returns None if loading fails, with a clear error message
#
# Test with the sample image AND with a path that doesn't exist.

# YOUR CODE HERE
def load_image_info(path):
    pass


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def load_image_info(path):
#     img = cv2.imread(path)
#     if img is None:
#         print(f"Error: could not load '{path}'")
#         return None
#
#     h, w = img.shape[:2]
#     channels = img.shape[2] if img.ndim == 3 else 1
#     size_kb = os.path.getsize(path) / 1024 if os.path.exists(path) else 0
#
#     return {
#         'image':    img,
#         'path':     path,
#         'shape':    img.shape,
#         'dtype':    img.dtype,
#         'size_kb':  round(size_kb, 1),
#         'channels': channels,
#         'width':    w,
#         'height':   h,
#     }
#
# # Test with valid path
# info = load_image_info('../assets/images/sample_scene.jpg')
# if info:
#     for k, v in info.items():
#         if k != 'image':
#             print(f"  {k}: {v}")
#
# print()
# # Test with invalid path
# bad_info = load_image_info('nonexistent.jpg')